# CRUCIAL Agora example trading code

In [ ]:
import json
import random
import os
import shutil
import subprocess

import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

## Setting up API authentication

Download the `agora-client` executable for your platform and put it in the same folder as this notebook.

On the CRUCIAL Agora platform, go to the Profile page.

Click the 'Download biscuit' button, and put the downloaded file in the same folder as this notebook.

Set the variable below to the filename.

In [ ]:
biscuit_filename = "agora_biscuit_2020-01-01T00_00_00.000Z.txt"

We use the `agora-client` executable to generate a public/private key pair, and display the public key.

If you have already generated a key pair, it will show the one it previously created

Linux specific:

Setting this environment variable causes `agora-client` to save the key pair in `~/.config/agora-client/`.  Otherwise `agora-client` will use Linux's "secret storage service", which I struggled to get working

In [ ]:
os.environ["AGORA_FORCE_PRIVATE_KEY_FILE"] = "true"

Mac specific:
Your OS may prevent `agora-client` from running, and display a message "Apple cannot verify agora-client is free of malware..."

To allow `agora-client` to run in the future, go to System Settings > Privacy & Security, then in the Security section click 'Open Anyway'

In [ ]:
try:
    ans = subprocess.check_output(["./agora-client", "--biscuit-file", biscuit_filename, "pubkey", "new"], text=True)
    print(ans)
except subprocess.CalledProcessError as e:
    print(f"Command failed: {e.returncode}")

On the Profile page, copy the above string (starting with `ed25519/`) below the instruction to "Enter public key...", then press the "Add public key" button.

If this is successful then the new public key should appear, with an option to Expire key.  If this does not happen, it was unsuccessful (see below).

#### Note
If you have done this before and your login session has since expired on the Agora platform (roughly every 7 days), you will need to remove the existing public/private key pair from your system and generate a new one using the code above.

For Linux: in the next box we define a function that deletes the key pair (but we do not run it)

In [ ]:
def delete_existing_key_pair():
    agora_pubkey_folder = os.path.join(os.path.expanduser('~'), ".config", "agora-client", "api.crucialab.net")
    shutil.rmtree(agora_pubkey_folder)

If you want to run this, uncomment and run the following line of code:

In [ ]:
# delete_existing_key_pair()

For Mac you will need to delete the saved key pair on your system.

## Market authentication test

In [ ]:
try:
    ans = subprocess.check_output(["./agora-client", "--biscuit-file", biscuit_filename, "get", "me"], text=True)
    print(ans)
except subprocess.CalledProcessError as e:
    print(f"Command failed: {e.returncode}")

If things are set up correctly, the above code should show your user ID and the email address you log in to the Agora platform with.

## List visible markets
The below code lists the markets visible to the user

In [ ]:
def view_all_markets():
    try:
        ans = subprocess.check_output(["./agora-client", "--biscuit-file", biscuit_filename, "get", "market"], text=True)
    except subprocess.CalledProcessError as e:
        print(f"Command failed: {e.returncode}")
    return json.loads(ans)

In [ ]:
view_all_markets()

## Get prices from a market

In [ ]:
def get_market_prices(market_id):
    try:
        ans = subprocess.check_output(["./agora-client", "--biscuit-file", biscuit_filename, "get", "price", "--market-id", str(market_id)], text=True)
        return ans
    except subprocess.CalledProcessError as e:
        print(f"Command failed: {e.returncode}")

We will get the prices from the demo "2050 CO2 concentration" market, which has ID 106

In [ ]:
MKT_ID = 106
market_price_json = json.loads(get_market_prices(MKT_ID))

The order these are returned in is random, but if we sort by `outcome_id` these end up in order from left to right.

_This behaviour might not be guaranteed?_

In [ ]:
market_price_json = sorted(market_price_json, key=lambda x: x['outcome_id'])

In [ ]:
market_price_json

In [ ]:
market_prices = np.array([foo['price'] for foo in market_price_json])

`market_prices` is a NumPy array of the current market prices in the specified market

In [ ]:
market_prices

## Constructing our own fair prices

Recall that each unit of an outcome pays 1 credit if the outcome occurs, and 0 if it doesn't.  The price of an outcome therefore represents a market consensus on the probability of an outcome occurring.

Your fair prices are what you, an expert participant, think the probability is for each outcome in the market.

For this example, we will create some artificial fair prices.  However, for a real market, you would likely construct these somewhere else and save them to a file.

In [ ]:
def generate_pretend_CO2_fair_prices():
    """
    We use a normal distribution with a randomly-generated mean and standard deviation.

    This function saves the prices out to file at fake_CO2_prices.txt
    """
    MEAN = random.uniform(400, 700)
    STDEV = random.uniform(50, 100)
    print(f"Generating fair prices for normal distribution with mean {MEAN:.1f} and stdev {STDEV:.1f}")

    nd = sp.stats.norm(loc=MEAN, scale=STDEV)
    cdfs = [0.0]
    for foo in range(300, 810, 50):
        cdfs.append(nd.cdf(foo))
    cdfs.append(1.0)
    cdfs = np.array(cdfs)

    prices = cdfs[1:] - cdfs[:-1]
    np.savetxt("fake_CO2_prices.txt", prices)

In [ ]:
generate_pretend_CO2_fair_prices()

We now load in the fair prices that the function saved.  For a real market, you would load in your own fair prices instead.

In [ ]:
fair_prices = np.loadtxt("fake_CO2_prices.txt")
print(fair_prices)

## Plotting fair prices and current market prices

In [ ]:
xvals = np.arange(275, 825.01, 50)
plt.plot(xvals, market_prices, label="market prices")
plt.plot(xvals, fair_prices, label="fair prices")
plt.legend()
plt.xlabel("CO2 concentration")
plt.ylabel("Price")
plt.grid()
plt.show()

## Preparing a trade using the market prices and our fair prices

Given an array of current market prices and an array of our fair prices, how should we act?

A reasonable starting point is to make a trade that maximises expected value (according to our fair prices) for the amount of credits that we are willing to spend.

There are caveats with this approach, which we will discuss later, but this is what we will implement in code.

To aid us, we will define some functions that implement the market pricing algorithm.  For more details on this, see the AGORA Trading Guide.

In [ ]:
def cost(b, q):
    """
    C(q) as defined in the Trading Guide, where q is the array of exposure of the AMM to the market outcomes.

    This is calculated as b*log(sum_i(e^(q_i/b))).
    
    This implementation includes the extra step from CRUCIAL's R example code to avoid overflow effects when q/b is large.
    """
    x = q/b
    x_max = x.max()
    x -= x_max
    return b * (x_max + np.log(np.exp(x).sum()))

In [ ]:
def prices_to_q(b, pri):
    """
    Converts an array of prices into q, the array of exposure of the AMM to outcomes.

    Note that this can only be done up to a constant, since a participant buying 1 of each outcome does not change the prices.

    This function assumes the lowest-priced outcome has q_i = 0.
    """
    return b*np.log(pri/pri.min())

We define a function that retrieves the liquidity factor for a specific market.  The liquidity factor determines how quickly the price moves in response to outcomes being bought (larger liquidity factor = less price movement).

In [ ]:
def get_liquidity_factor(mkt_id):
    """Returns the liquidity factor (b) for the appropriate market"""
    all_market_info = view_all_markets()
    for market in all_market_info:
        if market['market_id'] == mkt_id:
            return market['liquidity_factor']
    raise RuntimeError(f"Market ID {mkt_id} not found")

The `make_trade_amounts` algorithm produces trade quantities that maximise expected value for the given budget spend.

Intuitively, this is equivalent to greedily buying the outcomes with the largest ratio of (fair price / current price) until the budget is used up.

In [ ]:
def make_trade_amounts(b, current_prices, fair_prices, budget):
    """
    Implementing the optimal-EV trade algorithm as in CRUCIAL's R example code

    b: liquidity factor of the market
    current_prices: current market prices
    fair_prices: your model prices
    budget: max budget to spend
    """
    assert np.isclose(current_prices.sum(), 1.0), "Current prices must sum to 1"
    assert np.isclose(fair_prices.sum(), 1.0), "Fair prices must sum to 1"
    assert fair_prices.min() > 0.0, "Fair prices must be strictly positive"

    q_current = prices_to_q(b, current_prices)
    q_target = prices_to_q(b, fair_prices)
    delta_q = q_target - q_current
    delta_q -= delta_q.min()  # make non-negative

    full_cost = cost(b, q_current + delta_q) - cost(b, q_current)

    def fn(offset):
        """Function used by the root-finding algorithm.  The desired root is when this function outputs 0.

        The input scalar 'offset' is subtracted from every outcome, with a floor of 0 imposed
        """
        w = delta_q - offset
        w[w < 0.0] = 0.0
        return cost(b, q_current + w) - cost(b, q_current) - budget

    if budget < full_cost:
        sol = sp.optimize.root_scalar(fn, bracket=[0, delta_q.max()])
        if not sol.converged:
            raise RuntimeError(f"Root finding routine did not converge. {sol}")
        w = delta_q - sol.root
        w[w < 0.0] = 0.0
    else:
        w = delta_q

    w = w.astype(np.int32)  # buy quantities must be integers

    actual_cost = cost(b, q_current + w) - cost(b, q_current)
    print(f"Actual cost: {actual_cost:.3f}")
    return w

Let's see what quantities are proposed if we are willing to spend 20 credits on the demo market...

In [ ]:
BUDGET = 20.0
trade_quantities = make_trade_amounts(get_liquidity_factor(MKT_ID),
                                      market_prices,
                                      fair_prices,
                                      BUDGET)
print(trade_quantities)

## Performing the trade on the market

We now want to buy these quantities of the outcomes on the market.

On the CRUCIAL AGORA platform, we don't buy/sell outcomes directly.  Instead we must create contracts covering one or more outcomes, and buy/sell contracts.

Since the `make_trade_amounts` function proposes different quantities of each outcome, we will make separate contracts covering one outcome each.

A simple version of this script could simply make the contracts and then buy them.  This has the downside that, if we were to run the script several times, multiple contracts covering the same outcome would be generated, which is quite unwieldy.

We will introduce some extra complexity by fetching the current contracts held by the user on the market.  If we have previously made a relevant contract then we will re-use it.

### Defining more utility functions

In [ ]:
def generate_outcomes(market_id):
    """
    This is a utility function that, for a given market, generates a dictionary
    mapping outcome IDs to reasonable contract names.  This is used when creating
    contracts to trade.
    """
    mp_json = json.loads(get_market_prices(market_id))
    mp_json = sorted(mp_json, key=lambda x: x['outcome_id'])
    ids2names = {}
    for x in mp_json:
        onames = []
        # x['values'] has length 1 for 1D markets, 2 for 2D markets
        for xval in x['values']:
            # The variable can be either categorical or continuous
            if isinstance(xval, str):
                # categorical
                onames.append(xval)
            elif isinstance(xval, list):
                # continuous
                if xval[0] is None:
                    oname = f"Under {xval[1]}"
                elif xval[1] is None:
                    oname = f"Over {xval[0]}"
                else:
                    oname = f"{xval[0]} to {xval[1]}"
                onames.append(oname)
            else:
                raise Exception(f"Unsure how to handle x['values'][0] of type {type(xval)}")
        # for 1D markets, use the given name for the contract
        # for 2D markets, insert a comma between the given sub-names
        ids2names[x['outcome_id']] = "XYZ " + ", ".join(onames)
    return ids2names

In [ ]:
id2cname = generate_outcomes(MKT_ID)

In [ ]:
id2cname

In [ ]:
def convert_value_to_tuple(value):
    """
    Agora outcome 'values' are (nested) lists, which are not valid dictionary keys
    in Python.  This function turns them into (nested) tuples.
    """
    return tuple(tuple(foo) if isinstance(foo, list) else foo for foo in value)

In [ ]:
def generate_values_to_ids(market_id):
    """
    Utility function returning a dictionary mapping 'tupled' values to outcome IDs
    """
    mp_json = json.loads(get_market_prices(market_id))
    mp_json = sorted(mp_json, key=lambda x: x['outcome_id'])
    values2ids = {}
    for x in mp_json:
        tuple_value = convert_value_to_tuple(x['values'])
        values2ids[tuple_value] = x['outcome_id']
    return values2ids

In [ ]:
values2ids = generate_values_to_ids(MKT_ID)

In [ ]:
# values2ids

In [ ]:
outcome_ids = [foo['outcome_id'] for foo in market_price_json]

In [ ]:
# outcome_ids

In [ ]:
def get_contracts(market_id):
    """Function to get contracts that have been defined on a given market"""
    try:
        ans = subprocess.check_output(["./agora-client", "--biscuit-file", biscuit_filename, "get", "contract", "--market-id", str(market_id)], text=True)
        return json.loads(ans)
    except subprocess.CalledProcessError as e:
        print(f"Command failed: {e.returncode}")

In [ ]:
existing_contracts = get_contracts(MKT_ID)

In [ ]:
existing_contracts

In [ ]:
contracts_to_use = {}

In [ ]:
# loop over existing contracts; if we can use it instead of creating a new contract
# then add to contracts_to_use dictionary
for contract in existing_contracts:
    if len(contract['outcomes']) > 1:
        print(f"contract_id {contract['contract_id']} has more than 1 outcome, skipping...")
        continue
    tupled_outcome = convert_value_to_tuple(contract['outcomes'][0])
    outcome_id = values2ids[tupled_outcome]
    if contract['contract'] == id2cname[outcome_id]:
        contracts_to_use[outcome_id] = contract['contract_id']
        print(f"contract_id {contract['contract_id']} corresponds to '{id2cname[outcome_id]}', reusing...")
    else:
        print(f"contract_id {contract['contract_id']} has name '{contract['contract']}' not '{id2cname[outcome_id]}', skipping...")

In [ ]:
contracts_to_use

Any other outcomes that we want to buy, we must create contracts for them

In [ ]:
def create_contract(market_id, contract_name, outcome_ids):
    """Function to create contract covering the provided outcome IDs"""
    contract = {'contract': contract_name,
                'market_id': market_id,
                'outcomes': outcome_ids}
    contract_json = json.dumps(contract)
    try:
        ans = subprocess.check_output(["./agora-client", "--biscuit-file", biscuit_filename, "post", "contract"], input=contract_json, text=True)
        return json.loads(ans)
    except subprocess.CalledProcessError as e:
        print(f"Command failed: {e.returncode}")

In [ ]:
for oid, quantity in zip(outcome_ids, trade_quantities):
    if quantity == 0:
        # no need to create contract
        continue
    fn_out = create_contract(MKT_ID, id2cname[oid], [oid])
    print(f"Created contract for outcome '{id2cname[oid]}'")
    contracts_to_use[oid] = fn_out[0]['contract_id']

In [ ]:
contracts_to_use

Now we are finally ready to make the trade...

Firstly we define a function to interact with the market, but we don't execute it yet

In [ ]:
def make_trade(market_id, trades):
    """
    Function to send trade to the market.
    
    market_id is the market ID
    trades is a list of (contract_id, amount to buy or sell) tuples
    """
    trade_list = []
    for trade in trades:
        cid, amount = trade
        trade_list.append({'contract_id': cid,
                           'contracts': amount,
                           'market_id': market_id})
    trade_json = json.dumps(trade_list)
    try:
        ans = subprocess.check_output(["./agora-client", "--biscuit-file", biscuit_filename, "post", "execution"], input=trade_json, text=True)
        return json.loads(ans)
    except subprocess.CalledProcessError as e:
        print(f"Command failed: {e.returncode}")

In [ ]:
trades_to_make = []
for oid, quantity in zip(outcome_ids, trade_quantities):
    if quantity == 0:
        continue
    trades_to_make.append((contracts_to_use[oid], int(quantity)))

In [ ]:
trades_to_make

This is a list of (contract ID, amount to buy) for the trade we are about to make

## Execute the below line of code to send the trade to the market.

In [ ]:
make_trade(MKT_ID, trades_to_make)

The function output gives information about the execution of the trade.

### A more condensed version would be...

In [ ]:
# biscuit_filename = "..."
# os.environ["AGORA_FORCE_PRIVATE_KEY_FILE"] = "true"
# MKT_ID = ...
# BUDGET = ...

# market_price_json = json.loads(get_market_prices(MKT_ID))
# market_price_json = sorted(market_price_json, key=lambda x: x['outcome_id'])
# market_prices = np.array([foo['price'] for foo in market_price_json])

# fair_prices = np.loadtxt("...")

# trade_quantities = make_trade_amounts(get_liquidity_factor(MKT_ID), market_prices, fair_prices, BUDGET)

# id2cname = generate_outcomes(MKT_ID)
# values2ids = generate_values_to_ids(MKT_ID)
# outcome_ids = [foo['outcome_id'] for foo in market_price_json]
# existing_contracts = get_contracts(MKT_ID)

# contracts_to_use = {}
# # loop over existing contracts; if we can use it instead of creating a new contract
# for contract in existing_contracts:
#     if len(contract['outcomes']) > 1:
#         print(f"contract_id {contract['contract_id']} has more than 1 outcome, skipping...")
#         continue
#     tupled_outcome = convert_value_to_tuple(contract['outcomes'][0])
#     outcome_id = values2ids[tupled_outcome]
#     if contract['contract'] == id2cname[outcome_id]:
#         contracts_to_use[outcome_id] = contract['contract_id']
#         print(f"contract_id {contract['contract_id']} corresponds to '{id2cname[outcome_id]}', reusing...")
#     else:
#         print(f"contract_id {contract['contract_id']} has name '{contract['contract']}' not '{id2cname[outcome_id]}', skipping...")

# for oid, quantity in zip(outcome_ids, trade_quantities):
#     if quantity == 0:
#         # no need to create contract
#         continue
#     fn_out = create_contract(MKT_ID, id2cname[oid], [oid])
#     print(f"Created contract for outcome '{id2cname[oid]}'")
#     contracts_to_use[oid] = fn_out[0]['contract_id']

# trades_to_make = []
# for oid, quantity in zip(outcome_ids, trade_quantities):
#     if quantity == 0:
#         continue
#     trades_to_make.append((contracts_to_use[oid], int(quantity)))

# make_trade(MKT_ID, trades_to_make)

# Closing comments

## 1. Fair prices

Advice: don't blindly trust your model that creates your fair prices.

If they are wildly different to the market prices, either you are a genius and can make a lot of money.  Or your code is wrong or you have misunderstood what you are trading on.  Tread carefully!

## 2. Risk / credit allocation

Maximising expected value is okay when you are only trading small fractions of your balance, but in general you may want to take a more nuanced approach.

Informally, it's unwise to make trades where there's a meaningful chance of losing a large proportion of your balance, even if the upside is large.  You may want to read up on concepts like the "Kelly criterion".

## 3. Selling

In this tutorial we defined a function that put credits into the market.  You may also want to sell over-priced holdings.

If you want to implement this, the utility functions in this notebook may be useful.  The `get_portfolio()` function may also be helpful: this returns how many of each contract you hold (although unfortunately this cannot be restricted to a specific market).

In [ ]:
def get_portfolio():
    """Function to get account holdings from CRUCIAL AGORA"""
    try:
        ans = subprocess.check_output(["./agora-client", "--biscuit-file", biscuit_filename, "get", "portfolio"], text=True)
        return json.loads(ans)
    except subprocess.CalledProcessError as e:
        print(f"Command failed: {e.returncode}")

In [ ]:
def get_balance():
    """Function to get account balance from CRUCIAL AGORA"""
    try:
        ans = subprocess.check_output(["./agora-client", "--biscuit-file", biscuit_filename, "get", "balance"], text=True)
        return json.loads(ans)
    except subprocess.CalledProcessError as e:
        print(f"Command failed: {e.returncode}")

## 4. 2D markets

This code works also works for 2D markets.  You will likely want to pass back and forth between a 2D array and 1D flattened array, using NumPy's `.flatten()` and `.reshape()`.  The `market_prices` produced by this code are a 1D flattened array.